<h1 align="center">Gaia Spectral Classifier</h1> 
  <p align="center">
     Introduction to Artificial Intelligence Main Course Output 2 [GDINTAI-MCO2]
    <br />

## I: Introduction to the Task and Dataset
### 1.1 Task and Motivation
Performs **Multiclass Classification** to predict a Main Sequence Star's spectral class (O, B, A, F, G, K, M) *excluding Wolf Rayet (WNh), and other exotic classes*.
using astrometric and photometric features from Gaia DR3 *excluding leaking columns*. Motivated by procedural star system or terrain generation in video games, as well as ESA Gaia's role to widespread Machine Learning use in Astronomy.

### 1.2 Repository And Scope
Code, Commit History, and Documentation are maintained in the [GitHub Repository](https://github.com/FrenzyyLumina/Gaia-Spectral-Classifier)


## II: Description of The Dataset
### 2.1 Overview and Collection
The dataset [dataGaia.csv](https://www.kaggle.com/datasets/realkiller69/gaia-data-set-for-stellar-classificationdr3) is a Gaia Data Release 3 (DR3) subset sourced at Kaggle from the European Space Agency (ESA) Gaia mission. Gaia observes stars as a space observatory using astrometry and photometry, while DR3 derives parameters and classifications used as targets. The subset contains 611,689 instances and 28 metadata fields.

### 2.2 Dataset Structure

In [ ]:
import numpy as np
import optuna
import matplotlib.pyplot as plt
import pandas as pd
import lime
from lime import lime_tabular
import random
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder



data_GaDR3 = pd.read_csv('data/dataGaia.csv')

### 2.3 Row and Column Structure
Each row represents a single star in the DR3 Catalog source with its spectral class label (`SpType-ELS`) and other attributes across **28 columns**.
Counts per spectral class in this dataset are sampled rather than raw proportions. (`O`: 26,016; `B`: 85,673; `A`, `F`, `G`, `K`, `M`: 100,000 each).
Leaking fields are excluded from inputs during training.

* **Target, Identifiers & Astrometry**
  * `SpType-ELS`: Estimated spectral class by Gaia. (`O`–`M`).
  * `Source`: Unique Gaia DR3 catalog identifier.
  * `RA_ICRS`, `DE_ICRS`: Right Ascension, Declination.
  * `Plx`: Trigonometric parallax of the celestial body in milliarcseconds.
  * `PM`, `pmRA`, `pmDE`: Proper motion in RA/Dec and Renormalised Unit Weight Error (astrometric fit quality).
  * `Dist`: Distance to the celestial object (inverse parallax) in parsecs.


* **Photometry, Colors & Astrophysical Parameters:**
  * `Gmag`, `BPmag`, `RPmag`, `GRVSmag`: Mean apparent magnitudes in G, BP, RP, and RVS bands.
  * `e_Gmag`, `e_BPmag`, `e_RPmag`, `e_GRVSmag`: Magnitude errors for G, BP, RP, and RVS bands.
  * `BP-RP`, `BP-G`, `G-RP`: Photometric color indices.
  * `pscol`: Pseudocolor estimated by Gaia in µm⁻¹.
  * `Teff`, `Rad`, `Lum-Flame`, `Mass-Flame`, `Age-Flame`, `z-Flame`: Effective temperature (K), radius (solar radii), luminosity (solar luminosities), mass (solar masses), age (Gyr), and redshift (km/s).

## III: List of Requirements
* **Python 3**
* **pandas**
* **numpy**
* **matplotlib**
* **scikitlearn**
* **Optuna**
* **LIME**


## IV: Data cleaning, preprocessing and EDA
### 4.1 Data Cleansing and Structure
The [dataGaia.csv](https://www.kaggle.com/datasets/realkiller69/gaia-data-set-for-stellar-classificationdr3) contains records of observations of ~611k stars.
`SpType-ELS` is the target and astrometric/photometric features (`Plx`, `pmRA`, `pmDE`, `PM`, `Gmag`, `BPmag`, `RPmag`, `BP-RP`, `BP-G`, `G-RP`) are used, excluding target-leaking features.

Inspecting the datasheet and excluding columns that could cause repeated instances, incoherent values, or errors when preparing training, verification, and test sets. 

In [ ]:
data_GaDR3.head()
data_GaDR3.info()
data_GaDR3.isnull().sum().sort_values(ascending=False)
data_GaDR3 = data_GaDR3.drop(columns=excluded)

### 4.2 Preprocessing
Post-inspection and data cleanup for incoherent values and columns, the cleansed dataset will be prepared for exploration. Preprocessing makes the spectral-class data consistent and encoded for model analysis.
Target-leaking features: `Source`, `Teff` are excluded.

`Dist` , `Rad` , `Lum-Flame`, `Mass-Flame`, `Age-Flame`, `z-Flame` are excluded as they are Gaia astrophysical estimates derived from stellar modeling, since including them risks leaking temperature, luminosity, or evolutionary clues that are not genuine features and prevents pure feature-based prediction.

In [ ]:
target = "SpType-ELS"

excluded = {
    "Source", "Teff", "Dist", "Rad", "Lum-Flame",
    "Mass-Flame", "Age-Flame", "z-Flame",
}

features = [col for col in data_GaDR3.columns if col not in excluded and col != target]

## V: Dataset Split

Dataset will be split in a ratio of 75/15/15, or:

**-75% Training**
**-15% Validation**
**-15% Testing**

To prevent data leakage, Dataset will be split before preprocessing features, and parameters will be learned only with training data.

In [ ]:
X = data_GaDR3.drop(columns=['SpType-ELS', 'Source'])
y = data_GaDR3['SpType-ELS']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

## VI: Model Training

## VII: Error Analysis
### 6.1 Data Quality and Validation
Assuring the quality of the data post-test set is valid before interpreting any errors, data analysis models have weaknesses and also cannot guarantee a 100% success. Machine learning models can create mistakes within coding or logic within the model and mistakes can also be included within the dataset itself. Error analysis checks for the results to be validated and following errors can be used to improve the model. 

## VIII: References
* **ESA Gaia DR3 Database**
    * https://www.cosmos.esa.int/web/gaia/dr3
* **ESA Gaia DR3 Database Schema**
    * https://gaia.aip.de/metadata/gaiadr3/